# Module 31 — Knowledge Engineering & Graph RAG

## Learning contract
Predict → Build → Try → Break → Debug → Measure → Improve → Defend

We will compare explicit graph evidence with semantic/vector retrieval and learn when GraphRAG is justified.

## Concept map
Evidence → Entity → Claim → Relation → Graph → Bounded traversal → Evidence path → RAG/Agent

**Invariant:** graph structure is evidence, not authorization.

In [ ]:
from app.graph import Entity, Edge, KnowledgeGraph
from app.retrieval import shortest_paths
g = KnowledgeGraph()
for e in [Entity('sup-1','supplier','Acme','t1'), Entity('con-1','contract','C-101','t1'), Entity('sys-1','system','Payments','t1'), Entity('team-1','team','Platform','t1')]: g.add_entity(e)
g.add_edge(Edge('sup-1','HAS_CONTRACT','con-1','contract-src','t1')) if False else None
print('BUILD: entities=', list(g.entities))

The simple reference graph requires provenance. Create a tiny claim-compatible edge fixture by using a non-empty source identifier. The exercise is intentionally explicit about evidence identity.

In [ ]:
g.add_edge(Edge('sup-1','HAS_CONTRACT','con-1','src-contract','t1'))
g.add_edge(Edge('con-1','GOVERNS','sys-1','src-policy','t1'))
g.add_edge(Edge('sys-1','OWNED_BY','team-1','src-owner','t1'))
print('TRY path=', shortest_paths(g,'sup-1','team-1','t1',3))

## BREAK 1 — tenant isolation
Attempt to add a cross-tenant relationship. Predict whether the graph should accept it.

In [ ]:
g.add_entity(Entity('evil','team','Other','t2'))
try:
    g.add_edge(Edge('sys-1','OWNED_BY','evil','src-cross','t1'))
except PermissionError as exc:
    print('BREAK/DEBUG:', exc)

## BREAK 2 — provenance
An edge without source evidence must not become active knowledge.

In [ ]:
try:
    g.add_edge(Edge('sys-1','DEPENDS_ON','team-1','','t1'))
except ValueError as exc:
    print('BREAK/DEBUG:', exc)

## BREAK 3 — unbounded traversal
The graph must have a hard traversal limit. This is both a performance and data-exposure control.

In [ ]:
try:
    g.neighbors('sup-1','t1',99)
except ValueError as exc:
    print('BREAK/DEBUG:', exc)

## MEASURE
For a real benchmark record entity-resolution precision, provenance coverage, Recall@K, MRR, multi-hop accuracy, traversal work, p95 latency and cost per verified task.

In [ ]:
path = shortest_paths(g,'sup-1','team-1','t1',3)[0]
print('MEASURE hops=', len(path.relations), 'provenance_coverage=1.0')
print('Path evidence:', path.provenance)

## Industry exercise
Cybersecurity: Asset → Vulnerability → Control → Owner. Build a labeled multi-hop set and compare vector-only, graph-only and hybrid retrieval.

Banking: Customer → Account → Transaction → Case → Policy. Prove tenant/customer scope before context assembly.

Enterprise IT: Service → Dependency → Incident → Team → Runbook. Diagnose an outage with an evidence path.

## DEBUG challenge
A graph answer is plausible but cannot cite a source. Diagnose whether the defect is entity resolution, edge creation, provenance attachment, traversal, evidence assembly or generation. Write the smallest failing test before fixing it.

## DEFEND / mastery gate
Explain why graph retrieval is justified for the chosen workload, show the security ordering, run one failure injection, and provide a reproducible vector-vs-graph-vs-hybrid benchmark.

**Mastery:** Know → Construct → Connect → Measure → Break → Defend → Explain.